# 02b. 일관성 점검 (사실 확인)

**판정하지 않는다.** CoT 안의 서로 다른 부분이 같은 사실을 같게 적고 있는지, 데이터셋에 중복·편중이 있는지 **건수와 샘플**만 확인한다.
품질 기준 판정은 03 단계(판정 함수)에서 한다.

| # | 항목 | 비교 대상 |
|---|---|---|
| 1 | 환경 | git이 원본을 추적하지 않는지 |
| 2 | 상품 등급 | cot3에 적힌 상품별 등급 vs 상품 JSON `risk_grade` |
| 3 | 성향별 허용 등급 | 투자성향(cot1/cot2) × cot2 허용 등급 집합 |
| 4 | 추천 상품명 | answer 본문 vs `product_names` |
| 5 | 메타 vs 본문 | 메타 age·gender vs question 본문 |
| 6 | 고령자 고위험 | 70대 이상 × 1~2등급 추천 |
| 7 | 중복 | question 완전·유사 중복 |
| 8 | 추천 편중 | 상품별 추천 횟수 |
| 9 | 문장 완결 | answer 끝 문장 부호 |

추출 함수는 `src/extract.py`, 상품명 매칭은 D10 확정 규칙(정확 → 정규화 → 보험 모상품명)을 따른다.

In [ ]:
# 라이브러리와 프로젝트 함수 불러오기
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.load_cot import load_cot, load_products
from src.extract import (build_product_index, match_product, product_grade, parse_allowed_grades,
                         grade_after_name, propensity_in, extract_age_decade, extract_gender,
                         meta_age_match, quoted_strings, ends_with_punct, norm_full,
                         near_duplicate_groups)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

In [ ]:
# 데이터 읽기: CoT 전체(Train+Valid), 상품 JSON(Training 폴더, D1)
cot = load_cot()
prod = load_products()
idx = build_product_index(prod)
print("CoT:", cot.shape, "/ 상품:", prod.shape)

In [ ]:
# 추천 상품명을 한 행에 하나씩 펼치고 D10 규칙으로 상품 JSON에 매칭
rec = (cot[["split", "query_group", "category", "cot_id", "age", "product_names"]]
       .explode("product_names").dropna(subset=["product_names"])
       .rename(columns={"product_names": "rec_name"}).reset_index(drop=True))
m = rec["rec_name"].map(lambda n: match_product(n, idx))
rec["match_stage"] = m.str[0]
rec["keys"] = m.str[1]
rec["json_grade"] = rec["keys"].map(lambda k: product_grade(k, idx))
print("추천 상품명 수:", len(rec))
rec["match_stage"].value_counts()

## 1. 환경: git이 원본을 추적하지 않는지

In [ ]:
# data/raw/ 아래 파일이 .gitignore로 제외되는지 확인 (제외되면 규칙과 파일명이 출력됨)
r = subprocess.run(["git", "-C", str(ROOT), "check-ignore", "-v", "data/raw/Training/02.라벨링데이터/TL_2.소비자.zip"],
                   capture_output=True, text=True, encoding="utf-8")
print("check-ignore:", r.stdout.strip() or "(제외되지 않음)")

# git이 추적 중이거나 추가 대상으로 보는 파일 중 data/ 아래 파일 수
r = subprocess.run(["git", "-C", str(ROOT), "-c", "core.quotepath=false", "status", "--porcelain", "-uall"],
                   capture_output=True, text=True, encoding="utf-8")
tracked_data = [l for l in r.stdout.splitlines() if " data/" in l]
print("git status에 나온 data/ 파일 수:", len(tracked_data))

# 노트북 출력 제거 필터(nbstripout) 설정 확인
print(open(ROOT / ".gitattributes", encoding="utf-8").read())

## 2. cot3에 적힌 상품별 등급 vs 상품 JSON `risk_grade` (증권)

- 샘플 확인 결과 cot2에는 상품명이 나오지 않아(0%), 상품별 등급은 **cot3**에서 뽑는다.
- 범위 표현("N등급 이상/이하", "N~M등급", "N·M등급")은 제외한다.
- 추출 실패는 "상품명 뒤에 등급 표현 자체가 없음"과 "있지만 추출 실패(범위 표현만 있음)"로 나눈다.

In [ ]:
# 증권 추천 상품별로 cot3에서 등급을 뽑는다
sec = rec[rec["category"] == "증권"].merge(cot[["cot_id", "split", "cot3", "product_names"]], on=["cot_id", "split"])
res = sec.apply(lambda r: grade_after_name(r["cot3"], r["rec_name"], r["product_names"]), axis=1)
sec["cot3_status"] = res.str[0]
sec["cot3_grade"] = res.str[1]
sec["cot3_status"].value_counts().to_frame("건수")

In [ ]:
# 추출 성공 건에서 JSON 등급과 비교
# 비교 불가: 상품 JSON 매칭 실패 또는 이름 중복으로 등급이 여러 개인 경우
ok = sec[sec["cot3_status"] == "추출"].copy()
ok["비교"] = np.select(
    [ok["json_grade"].isna(), ok["cot3_grade"] == ok["json_grade"]],
    ["비교 불가(JSON 등급 없음)", "일치"], default="불일치")
print(ok["비교"].value_counts())
print()
# 불일치 건: cot3 등급 × JSON 등급 교차표
mis = ok[ok["비교"] == "불일치"]
pd.crosstab(mis["cot3_grade"], mis["json_grade"], margins=True)

In [ ]:
# 불일치 샘플 10건: 상품명 뒤 구간의 앞 120자를 함께 본다
def after_name(text, name, n=120):
    i = str(text).find(name)
    return str(text)[i + len(name): i + len(name) + n] if i >= 0 else ""

s = mis.sample(min(10, len(mis)), random_state=0)
for _, r in s.iterrows():
    print(f"[{r['split']} cot_id={r['cot_id']}] {r['rec_name']}")
    print(f"   cot3 등급={r['cot3_grade']:.0f} / JSON 등급={r['json_grade']:.0f}")
    print("   상품명 뒤:", after_name(r["cot3"], r["rec_name"]).replace("\n", " "))

In [ ]:
# 추출 실패 샘플: '등급 표현 없음' 5건, '있지만 추출 실패' 5건 (상품명 뒤 150자)
for status in ["등급 표현 없음", "있지만 추출 실패"]:
    sub = sec[sec["cot3_status"] == status]
    print("=" * 100)
    print(status, f"({len(sub)}건)")
    for _, r in sub.sample(min(5, len(sub)), random_state=0).iterrows():
        print(f"[cot_id={r['cot_id']}] {r['rec_name']}")
        print("   상품명 뒤:", after_name(r["cot3"], r["rec_name"], 150).replace("\n", " "))

## 3. 투자성향별 cot2 허용 등급 표기

- 성향은 cot1 기준과 cot2 기준을 나눠 뽑는다 (증권_03은 cot2에 성향이 나온다).
- 성향 명칭이 2개 이상이면 '복수', 없으면 '없음'으로 둔다.

In [ ]:
# 증권 CoT만 대상 (보험 cot2에는 등급 표현이 거의 없음: 02 노트북 6-3)
sc = cot[cot["category"] == "증권"].copy()
sc["성향_cot1"] = sc["cot1"].map(propensity_in)
sc["성향_cot2"] = sc["cot2"].map(propensity_in)
parsed = sc["cot2"].map(parse_allowed_grades)
sc["허용등급"] = parsed.str[0].map(lambda s: "-".join(map(str, sorted(s))) if s else "추출 실패")
sc["허용등급_출처"] = parsed.str[1]
print(sc["허용등급_출처"].value_counts())

In [ ]:
# cot1 성향 × 허용 등급 집합
pd.crosstab(sc["성향_cot1"], sc["허용등급"], margins=True)

In [ ]:
# cot2 성향 × 허용 등급 집합
pd.crosstab(sc["성향_cot2"], sc["허용등급"], margins=True)

In [ ]:
# 같은 성향(cot1 또는 cot2에서 하나만 나온 경우)인데 허용 등급 집합이 몇 종류인지
sc["성향_대표"] = np.where(sc["성향_cot1"].isin(["없음", "복수"]), sc["성향_cot2"], sc["성향_cot1"])
one = sc[~sc["성향_대표"].isin(["없음", "복수"]) & (sc["허용등급"] != "추출 실패")]
summary = one.groupby("성향_대표")["허용등급"].agg(
    건수="size", 허용등급_종류수="nunique", 최빈값=lambda s: s.value_counts().index[0],
    최빈값_비율=lambda s: round(s.value_counts(normalize=True).iloc[0] * 100, 1))
summary

In [ ]:
# 증권_03 중 cot1에 성향이 없는 건 → cot2에 성향이 있는지
s3 = sc[sc["query_group"] == "증권_03.상품 비교 추천형"]
no1 = s3[s3["성향_cot1"] == "없음"]
print("증권_03 전체:", len(s3), "/ cot1에 성향 없음:", len(no1))
no1["성향_cot2"].value_counts().to_frame("cot2 성향")

## 4. answer 속 상품명 vs `product_names`

두 가지 방법으로 본다.
- (A) answer에서 큰따옴표 안의 문자열을 뽑아 `product_names`와 비교
- (B) `product_names`의 각 상품명(정규화)이 answer 본문(정규화)에 포함되는지

In [ ]:
# (A) 따옴표 추출
ans = cot[["split", "query_group", "cot_id", "answer", "product_names"]].copy()
ans["quoted"] = ans["answer"].map(quoted_strings)
ans["list_norm"] = ans["product_names"].map(lambda L: {norm_full(x) for x in L})
ans["quoted_norm"] = ans["quoted"].map(lambda L: {norm_full(x) for x in L})
# answer 따옴표에만 있는 이름 (목록에 없음)
ans["only_answer"] = ans.apply(lambda r: r["quoted_norm"] - r["list_norm"], axis=1)
# 목록에는 있으나 answer 따옴표에 없는 이름
ans["only_list"] = ans.apply(lambda r: r["list_norm"] - r["quoted_norm"], axis=1)
print("따옴표가 하나도 없는 answer:", (ans["quoted"].map(len) == 0).sum())
print("answer 따옴표에만 있는 이름이 1개 이상인 CoT:", (ans["only_answer"].map(len) > 0).sum())
print("목록에만 있는 이름이 1개 이상인 CoT:", (ans["only_list"].map(len) > 0).sum())

In [ ]:
# (B) product_names 각 상품명이 answer 본문에 포함되는지 (둘 다 정규화)
ans["answer_norm"] = ans["answer"].map(norm_full)
ans["list_missing"] = ans.apply(lambda r: [x for x in r["product_names"] if norm_full(x) not in r["answer_norm"]], axis=1)
ans["n_missing"] = ans["list_missing"].map(len)
print("목록 상품 중 answer 본문에 없는 상품이 1개 이상인 CoT:", (ans["n_missing"] > 0).sum())
pd.crosstab(ans["query_group"], ans["n_missing"])

In [ ]:
# 샘플: (A) answer 따옴표에만 있는 이름 5건, (B) 본문에 없는 목록 상품 5건
print("=== (A) answer 따옴표에만 있는 이름 ===")
for _, r in ans[ans["only_answer"].map(len) > 0].sample(5, random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 따옴표 원문:", [q for q in r["quoted"] if norm_full(q) in r["only_answer"]][:2])
print()
print("=== (B) answer 본문에 없는 목록 상품 ===")
sub = ans[ans["n_missing"] > 0]
for _, r in sub.sample(min(5, len(sub)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 없는 상품:", r["list_missing"][:2])

## 5. 메타 age·gender vs question 본문

In [ ]:
# 본문에서 연령대·성별 추출 후 메타와 비교
q = cot[["split", "query_group", "category", "cot_id", "age", "gender", "question"]].copy()
q["본문_연령대"] = q["question"].map(extract_age_decade)
q["연령_비교"] = [meta_age_match(a, d) for a, d in zip(q["age"], q["본문_연령대"])]
q["본문_성별"] = q["question"].map(extract_gender)
q["성별_비교"] = np.select([q["본문_성별"].isna(), q["본문_성별"] == "복수", q["본문_성별"] == q["gender"]],
                         ["본문에 없음", "본문에 복수", "일치"], default="불일치")
print(pd.crosstab(q["query_group"], q["연령_비교"], margins=True))
print()
print(pd.crosstab(q["query_group"], q["성별_비교"], margins=True))

In [ ]:
# 불일치 샘플 (연령 5건, 성별 5건): 본문에서 표현 주변 60자
import re
def around(text, pat, n=30):
    m = re.search(pat, str(text))
    return str(text)[max(0, m.start() - n): m.end() + n] if m else ""

print("=== 연령 불일치 ===")
for _, r in q[q["연령_비교"] == "불일치"].sample(5, random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 메타={r['age']} / 본문:", around(r["question"], r"[1-9]0\s*대|\d{2}\s*(?:세|살)"))
print("=== 성별 불일치 ===")
sub = q[q["성별_비교"] == "불일치"]
for _, r in sub.sample(min(5, len(sub)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 메타={r['gender']} / 본문:", around(r["question"], r"남성|남자|여성|여자"))

## 6. 70대 이상 고객 CoT 중 1~2등급 상품 추천 (증권)

- 메타 age가 `70대`, `70대 이상`, `80대 이상`인 CoT.
- 등급은 상품 JSON 기준. 보험 상품에는 등급이 없어 증권만 본다.

In [ ]:
# 고령 메타 연령 × 증권 추천 상품의 JSON 등급
OLD = ["70대", "70대 이상", "80대 이상"]
old = rec[(rec["category"] == "증권") & rec["age"].isin(OLD)].copy()
print("대상 CoT 수:", old["cot_id"].nunique(), "/ 추천 상품 수:", len(old))
print(pd.crosstab(old["age"], old["json_grade"].fillna(-1).astype(int).replace(-1, "등급 없음/불명"), margins=True))

hi = old[old["json_grade"].isin([1, 2])]
print()
print("1~2등급 상품이 1개 이상 추천된 CoT 수:", hi["cot_id"].nunique())

In [ ]:
# 샘플 10건: 메타 연령, 성향(cot1/cot2), cot2 허용 등급, 추천 상품과 등급
hs = hi.merge(sc[["cot_id", "성향_cot1", "성향_cot2", "허용등급"]], on="cot_id", how="left")
hs.drop_duplicates("cot_id").sample(min(10, hi["cot_id"].nunique()), random_state=0)[
    ["cot_id", "age", "성향_cot1", "성향_cot2", "허용등급", "rec_name", "json_grade"]]

## 7. question 중복

- 완전 중복: 앞뒤 공백 제거·연속 공백 통일 후 문장이 똑같은 경우
- 유사 중복: 글자 2~3그램 TF-IDF 코사인 유사도 0.9 이상을 연결한 묶음 (500행 블록 단위 계산)

In [ ]:
# 완전 중복
q["q_norm"] = q["question"].str.strip().str.replace(r"\s+", " ", regex=True)
grp = q.groupby("q_norm")
dup_groups = grp.filter(lambda g: len(g) > 1)
print("완전 중복 묶음 수:", dup_groups["q_norm"].nunique(), "/ 포함 행 수:", len(dup_groups))
print("Train↔Valid에 걸친 완전 중복 묶음 수:", dup_groups.groupby("q_norm")["split"].nunique().gt(1).sum())

In [ ]:
# 유사 중복 (시간이 조금 걸린다)
q["near_group"] = near_duplicate_groups(q["question"].tolist(), threshold=0.9, block=500)
sizes = q.groupby("near_group").size()
multi = sizes[sizes > 1]
print("유사 중복 묶음 수(2건 이상):", len(multi), "/ 포함 행 수:", multi.sum())
print("최대 묶음 크기:", multi.max() if len(multi) else 0, "/ 10건 이상 묶음 수:", (multi >= 10).sum())
print()
print("묶음 크기 분포:")
print(multi.value_counts().sort_index().rename("묶음 수").to_frame().T)

In [ ]:
# 묶음이 어느 split에 걸쳐 있는지: Train만 / Valid만 / Train↔Valid
splits = q[q["near_group"].isin(multi.index)].groupby("near_group")["split"].agg(lambda s: "↔".join(sorted(set(s))))
splits.value_counts().rename("묶음 수").to_frame()

In [ ]:
# 가장 큰 유사 중복 묶음의 질문 3개 앞 120자
if len(multi) == 0:
    print("유사 중복 묶음이 없다.")
else:
    big = multi.idxmax()
    for t in q.loc[q["near_group"] == big, "question"].head(3):
        print("-", t[:120])

In [ ]:
# 참고: 임계값을 낮추면 묶음이 얼마나 생기는지 (0.9 결과를 해석할 때 비교용, 판정 아님)
for th in [0.8, 0.7]:
    lab = near_duplicate_groups(q["question"].tolist(), threshold=th, block=500)
    s = pd.Series(lab).value_counts()
    s = s[s > 1]
    print(f"유사도 {th} 이상: 묶음 {len(s)}개, 포함 행 {s.sum()}, 최대 크기 {s.max() if len(s) else 0}")

# 각 질문의 '가장 비슷한 다른 질문' 유사도 분포 (블록 계산)
from sklearn.feature_extraction.text import TfidfVectorizer
X = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit_transform(q["question"])
best = []
for s0 in range(0, X.shape[0], 500):
    sim = (X[s0:s0 + 500] @ X.T).toarray()
    for i in range(sim.shape[0]):
        sim[i, s0 + i] = 0          # 자기 자신 제외
    best.append(sim.max(axis=1))
best = np.concatenate(best)
pd.Series(best).describe(percentiles=[.5, .9, .99]).round(3)

## 8. 상품별 추천 횟수

In [ ]:
# 매칭된 상품 키 기준 추천 횟수
# 애매한 경우 처리: 이름 중복으로 후보가 여러 개면 대표 이름(display_name) 단위로 센다
tbl = idx["table"]
rec["display_name"] = rec["keys"].map(lambda k: tbl.loc[k[0], "display_name"] if k else None)
cnt = rec.dropna(subset=["display_name"]).groupby(["category", "display_name"]).size().rename("추천 횟수")
cnt.sort_values(ascending=False).head(20).to_frame()

In [ ]:
# 한 번도 추천되지 않은 상품 수 (상품 JSON 10,000개 기준)
used_keys = set(k for ks in rec["keys"] for k in ks)
tbl2 = tbl.assign(추천됨=tbl.index.isin(used_keys))
out = tbl2.groupby("domain")["추천됨"].agg(전체="size", 추천됨="sum")
out["추천 안 됨"] = out["전체"] - out["추천됨"]
print(out)
print()
print("주의: 매칭 실패(판정불가) 추천명", (rec["keys"].map(len) == 0).sum(), "건은 어떤 상품에도 연결되지 않아, 일부 상품이 '추천 안 됨'으로 세어졌을 수 있다.")

In [ ]:
# 상품 분류별 추천 안 된 비율
g = tbl2.groupby(["domain", "product_group"])["추천됨"].agg(전체="size", 추천됨="sum")
g["추천 안 된 비율(%)"] = ((1 - g["추천됨"] / g["전체"]) * 100).round(1)
g

## 9. answer가 문장 부호 없이 끝나는 건

In [ ]:
# 끝 공백·닫는 따옴표/괄호를 건너뛴 마지막 글자가 . ! ? … 。 가 아닌 건
ans["ends_ok"] = ans["answer"].map(ends_with_punct)
print("문장 부호 없이 끝나는 answer:", (~ans["ends_ok"]).sum())
print(pd.crosstab(ans["query_group"], ans["ends_ok"]))

In [ ]:
# 샘플 10건: 끝 60자
for _, r in ans[~ans["ends_ok"]].sample(min(10, (~ans["ends_ok"]).sum()), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] ...{r['answer'][-60:]!r}")

## 02b 정리 (직접 작성)

1. 2번 등급 불일치는 어떤 패턴인가? 03 판정(기준 3)에서 상품 등급의 기준을 무엇으로 할 것인가? (cot3 표기 / JSON)
2. 3번에서 같은 성향인데 허용 등급이 다른 경우를 어떻게 볼 것인가?
3. 4·5·9번 불일치·미완결 건을 정제 데이터셋에서 어떻게 다룰 것인가?
4. 6번 고령자 고위험 추천을 품질 기준에 넣을 것인가? 근거 규정은 무엇인가? (docs/sources.md)
5. 7번 중복을 Train/Valid 구성과 정제 데이터셋에서 어떻게 처리할 것인가?
6. decisions.md에 추가할 항목:
